# 28 · Sampling (server asks the host's LLM)

**Sampling** inverts the usual direction: a *server* asks the *client* to run
an LLM completion via a `sampling/createMessage` request. This lets a server
be agentic (summarize, classify, plan) **without holding any API keys** — the
host owns the model, the user stays in control, and cost stays on the host.

We use the in-process `Session` so the round-trip runs in one cell; the client
provides the model through `get_provider`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # PHASE_4_MCP root
import mcp_lab
print('mcp_lab loaded; protocol', mcp_lab.PROTOCOL_VERSION)

In [ ]:
from study_buddy import get_provider
PROVIDER = 'ollama'  # swap: openai | deepseek | anthropic | oxalpha | ollama
llm = get_provider(PROVIDER)
print('using provider:', PROVIDER)

In [ ]:
from mcp_lab.advanced import Session

# The CLIENT supplies the sampling capability, backed by any provider.
def sampling_handler(params):
    prompt = params['messages'][0]['content']['text']
    try:
        text = llm.chat([{'role':'user','content':prompt}]).text
    except Exception as e:
        text = f'(LLM unavailable: {e})'
    return {'role':'assistant','content':{'type':'text','text':text}}

session = Session(sampling_handler=sampling_handler)

@session.tool(description='Summarize text using the host LLM.')
def summarize(ctx, text):
    return ctx.sample('Summarize in one sentence:\n' + text)

print(session.call_tool('summarize', {'text':'MCP lets servers expose tools, resources, and prompts to any host.'}))

Key point: the server never imported a provider. It asked the client, which
chose the model. Swap `PROVIDER` to route sampling through a different LLM.